In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from utils import (
    OCTMendeleyDataset, 
    plotimg, 
    LitSupervised, 
    OCTRandAugment, 
    TransferLearning, 
    GLOBAL_CONFIG,
    product_dict,
    MAGNITUDE_MAX_RANGE,
    stringify_map,
    LOGDIR,
)
import torch
from torch import nn
from torchvision.models import (
    resnet18,
    ResNet18_Weights,
    resnet34,
    ResNet34_Weights,
    resnet50,
    ResNet50_Weights,
    resnet152,
    ResNet152_Weights,
    densenet121,
    DenseNet121_Weights,
    densenet161,
    DenseNet161_Weights,
    densenet169,
    DenseNet169_Weights,
    densenet201,
    DenseNet201_Weights,
)
from torchvision.transforms import v2
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from torchinfo import summary

In [ ]:
train_dataset = OCTMendeleyDataset("train")
validation_dataset = OCTMendeleyDataset("validation")
test_dataset = OCTMendeleyDataset("test")

In [ ]:
OCTMendeleyDataset.classes()

In [ ]:
_index = 6
path, _ = test_dataset.path_instance_pair(_index)
instance, label = test_dataset[_index]

In [ ]:
plotimg(instance, f"{OCTMendeleyDataset.classes()[label]} {path}")

In [ ]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=GLOBAL_CONFIG["batch_size"], 
    num_workers=7,
)
validation_loader = DataLoader(
    validation_dataset, 
    batch_size=GLOBAL_CONFIG["batch_size"], 
    num_workers=7,
)

In [ ]:
TransferLearning.models = {
    "resnet18": resnet18(weights = ResNet18_Weights.IMAGENET1K_V1),
    #"resnet34": resnet34(weights = ResNet34_Weights.IMAGENET1K_V1),
    #"resnet50": resnet50(weights = ResNet50_Weights.IMAGENET1K_V2),
    #"resnet152": resnet152(weights = ResNet152_Weights.IMAGENET1K_V2),
    "densenet121": densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1),
    "densenet161": densenet161(weights=DenseNet161_Weights.IMAGENET1K_V1),
    "densenet169": densenet169(weights=DenseNet169_Weights.IMAGENET1K_V1),
    "densenet201": densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1),
    #"efficientnet_b0": efficientnet_b0(),
    #"mobilenet_v2": mobilenet_v2(),
    #"mobilenet_v3_large": mobilenet_v3_large(),
    #"mobilenet_v3_small": mobilenet_v3_small(),
}

In [ ]:
size = instance.size
summary(TransferLearning.models["densenet121"], input_size=(GLOBAL_CONFIG["batch_size"],size(0),size(1),size(2)), depth = 2)

In [ ]:
TransferLearning.replace_head("densenet121", len(OCTMendeleyDataset.classes()))

In [ ]:
size = instance.size
summary(TransferLearning.models["densenet121"], input_size=(GLOBAL_CONFIG["batch_size"],size(0),size(1),size(2)), depth = 2)

In [ ]:
def train_model(configs, runname):
    LOGCONFIG = "config.txt"
    log_config_path = LOGDIR / Path(LOGCONFIG)

    last_config = None
    if log_config_path.exists():
        with open(log_config_path, "r") as fconfig:
            last_config = eval(fconfig.read())

    configarr = list(product_dict(configs))
    last_index = configarr.index(last_config) if isinstance(last_config, dict) else None
    
    current_config = None
    current_name = None
    try:
        for current_index, config in enumerate(configarr):
            if isinstance(last_index, int) and current_index < last_index:
                continue
              
            version_name = f"{runname}_{stringify_map(config)}"
            logger = TensorBoardLogger(
               save_dir=".",
               name=LOGDIR,
               version=version_name,
            )
            trainer = L.Trainer(
               logger=logger,
               accelerator="gpu",
               callbacks=EarlyStopping(monitor=LitSupervised.VALIDATION_LOSS, mode="min", patience=8),
            )
            litmodel = LitSupervised(TransferLearning.models["resnet152"], config)
            
            ckpt_path = None
            
            current_name = stringify_map(config, ", ")
            if current_index == last_index:
               print(f"Resuming at {current_name}")
               ckpts_path = Path(LOGDIR) / version_name / "checkpoints"
               if ckpts_path.exists():
                   ckpts_arr = sorted(list(ckpts_path.iterdir()))
                   ckpt_path = None if ckpts_arr == [] else ckpts_arr[-1]
            else:
               print(current_name)

            current_config = config
            trainer.fit(
               model=litmodel, 
               train_dataloaders=train_loader, val_dataloaders=validation_loader,
               ckpt_path=ckpt_path,
            )

    except KeyboardInterrupt:
        if isinstance(current_config, dict):
            print(f"Saving {current_name}...")
            with open(log_config_path, "w") as fconfig:
                fconfig.write(str(current_config))

In [ ]:
def training_step1(model_name):
    configs = {
        "magnitude": range(MAGNITUDE_MAX_RANGE),
        "lr": [1e-3],
    }
    # Freeze weights and train
    TransferLearning.set_grads(model_name)
    train_model(configs, "step1")
    

In [ ]:
def training_step2(model_name):
    configs = {
        "magnitude": range(MAGNITUDE_MAX_RANGE),
        "lr": [1e-4],
    }
    # Unfreeze weights and train
    TransferLearning.set_grads(model_name, fill=True)
    train_model(configs, "step2")

In [ ]:
training_step1("resnet152")